# Slowly Changing Dimensions (SCD): A Complete Guide

## The Problem

Imagine this scenario:
- A customer upgrades from **Basic** to **Premium** tier today
- Last month they were on **Basic**
- If your pipeline just overwrites their profile, your month-over-month revenue reports won't match active user counts

On the flip side:
- Someone corrects a typo in their email address
- If you track every typo historically, your database bloats with meaningless data

## The Solution: Slowly Changing Dimensions

**Slowly Changing Dimensions (SCDs)** are techniques for managing and maintaining dimensional data when attributes change over time.

Today we'll cover:
- **Type 1**: Overwrite old data (for corrections)
- **Type 2**: Preserve history with date ranges (for true changes)
- **Type 3**: Keep previous value in a separate column (limited history)
- **SCD2 Deletes**: How to handle deleted records historically
- **Production Pattern**: A single MERGE statement that handles both Type 1 and Type 2 logic

---

*This pattern works identically in Databricks Delta Lake and Azure SQL Database.*

## SCD Type 1: Overwrite

### What Is It?

**Type 1** means: **Overwrite the old data with new data. No history is kept.**

Think of it like updating a contact in your phone:
- Your friend gets a new phone number
- You don't create "Friend V2"
- You just open their contact, erase the old number, and type the new one

### When to Use Type 1

✅ **Use Type 1 when:**
- Correcting typos or data entry errors
- Business users only care about the *current* state
- Historical reporting is irrelevant for this attribute
- Example: Email address typo, phone number update

❌ **Don't Use Type 1 when:**
- Compliance requires an audit trail
- Analysts need historical reports
- Once overwritten, data is **gone forever**

### Visual Example

```
BEFORE:
CustomerID | Name  | Email
1          | Alice | alcie@example.com  ❌ (typo)

AFTER Type 1 UPDATE:
CustomerID | Name  | Email
1          | Alice | alice@example.com  ✅ (corrected)
```

The old email is **gone**. No history preserved.

## Setup: Create Sample Database and Tables

Let's create a demo environment to illustrate SCD patterns.

In [0]:
-- Create a demo catalog and schema for SCD examples
CREATE CATALOG IF NOT EXISTS scd_demo;
USE CATALOG scd_demo;

CREATE SCHEMA IF NOT EXISTS dimension;
USE SCHEMA dimension;

In [0]:
-- SCD Type 1: Simple customer table (no history tracking)
DROP TABLE IF EXISTS Customer_Type1;

CREATE TABLE Customer_Type1 (
  CustomerID INT PRIMARY KEY,
  Name STRING,
  Email STRING,
  Phone STRING,
  LastUpdated TIMESTAMP
);

-- Insert initial data
INSERT INTO Customer_Type1 VALUES
  (1, 'Alice Johnson', 'alcie@example.com', '555-0101', current_timestamp()),  -- Typo in email
  (2, 'Bob Smith', 'bob@example.com', '555-0102', current_timestamp()),
  (3, 'Carol White', 'carol@example.com', '555-0103', current_timestamp());

SELECT * FROM Customer_Type1;

### Type 1: Correcting Alice's Email

Alice noticed her email has a typo: `alcie@example.com` should be `alice@example.com`.

We'll use a simple `UPDATE` statement to overwrite the incorrect value.

In [0]:
-- Type 1: Overwrite the incorrect email
UPDATE Customer_Type1
SET Email = 'alice@example.com',
    LastUpdated = current_timestamp()
WHERE CustomerID = 1;

-- View the result: old email is GONE
SELECT * FROM Customer_Type1 WHERE CustomerID = 1;

## SCD Type 2: Preserve History

### What Is It?

**Type 2** means: **Keep a complete history of changes by creating new rows.**

Think of it like a car's title history:
- It's not enough to know who owns the car *now*
- You need a record of *all previous owners* and the exact dates they owned it

### How It Works

We add three special columns:
- **`ValidFrom`**: When this version of the record became active
- **`ValidTo`**: When this version of the record was superseded (NULL for current record)
- **`IsCurrent`**: Boolean flag (1 = current, 0 = historical)

### When to Use Type 2

✅ **Use Type 2 when:**
- Analysts need to report on historical states
- Compliance requires audit trails
- Revenue or metrics are tied to dimension state at a point in time
- Example: Subscription tier changes, pricing changes, organizational changes

❌ **Don't Use Type 2 when:**
- Tracking rapidly changing data (like LastLoginDate)
- This creates a "Rapidly Changing Dimension" anti-pattern
- Your table will bloat with millions of unnecessary rows

### Visual Example

```
BEFORE: Alice upgrades from Basic to Premium

CustomerID | Name  | Tier  | ValidFrom  | ValidTo    | IsCurrent
1          | Alice | Basic | 2025-01-01 | NULL       | 1

AFTER Type 2 UPDATE:

CustomerID | Name  | Tier    | ValidFrom  | ValidTo    | IsCurrent
1          | Alice | Basic   | 2025-01-01 | 2026-04-01 | 0  ← Historical
1          | Alice | Premium | 2026-04-01 | NULL       | 1  ← Current
```

We now have **TWO rows** for Alice, preserving the history of when she had each tier.

In [0]:
-- SCD Type 2: Customer table WITH history tracking
DROP TABLE IF EXISTS Customer_Type2;

CREATE TABLE Customer_Type2 (
  CustomerKey BIGINT GENERATED ALWAYS AS IDENTITY,  -- Surrogate key
  CustomerID INT,                                    -- Natural key
  Name STRING,
  Email STRING,
  SubscriptionTier STRING,
  ValidFrom DATE,
  ValidTo DATE,
  IsCurrent BOOLEAN,
  PRIMARY KEY (CustomerKey)
);

-- Insert initial data (all records are current)
INSERT INTO Customer_Type2 (CustomerID, Name, Email, SubscriptionTier, ValidFrom, ValidTo, IsCurrent)
VALUES
  (1, 'Alice Johnson', 'alice@example.com', 'Basic', '2025-01-01', NULL, TRUE),
  (2, 'Bob Smith', 'bob@example.com', 'Premium', '2025-01-15', NULL, TRUE),
  (3, 'Carol White', 'carol@example.com', 'Basic', '2025-02-01', NULL, TRUE);

SELECT * FROM Customer_Type2 ORDER BY CustomerID, ValidFrom;

### Type 2: Alice Upgrades to Premium

Alice upgrades from Basic to Premium tier on April 1, 2026.

**The Type 2 Process:**
1. **Close the old record**: Set `ValidTo = '2026-04-01'` and `IsCurrent = FALSE`
2. **Insert a new record**: Create a new row with `SubscriptionTier = 'Premium'`, `ValidFrom = '2026-04-01'`, `ValidTo = NULL`, `IsCurrent = TRUE`

In [0]:
-- Step 1: Close the current 'Basic' record
UPDATE Customer_Type2
SET ValidTo = '2026-04-01',
    IsCurrent = FALSE
WHERE CustomerID = 1 AND IsCurrent = TRUE;

-- View the result
SELECT * FROM Customer_Type2 WHERE CustomerID = 1 ORDER BY ValidFrom;

In [0]:
-- Step 2: Insert the new 'Premium' record
INSERT INTO Customer_Type2 (CustomerID, Name, Email, SubscriptionTier, ValidFrom, ValidTo, IsCurrent)
VALUES (1, 'Alice Johnson', 'alice@example.com', 'Premium', '2026-04-01', NULL, TRUE);

-- View the complete history
SELECT * FROM Customer_Type2 WHERE CustomerID = 1 ORDER BY ValidFrom;

### Querying Type 2 Tables

**Current State**: Use `WHERE IsCurrent = TRUE`

**Historical State (Point-in-Time)**: Use date range filtering

Example: "What tier was Alice on February 15, 2026?"

In [0]:
-- Get current state of all customers
SELECT CustomerID, Name, Email, SubscriptionTier, ValidFrom
FROM Customer_Type2
WHERE IsCurrent = TRUE
ORDER BY CustomerID;

In [0]:
-- What tier was Alice on February 15, 2026?
SELECT CustomerID, Name, SubscriptionTier, ValidFrom, ValidTo
FROM Customer_Type2
WHERE CustomerID = 1
  AND ValidFrom <= '2026-02-15'
  AND (ValidTo IS NULL OR ValidTo > '2026-02-15');

-- Answer: She was on 'Basic' tier (which started 2025-01-01 and ended 2026-04-01)

## SCD Type 2: Handling Deletes

### The Challenge

What happens when a customer closes their account? We have two options:

### Option 1: Soft Delete (Recommended for Type 2)

**Keep the record but mark it as deleted:**
- Set `ValidTo = <deletion_date>`
- Set `IsCurrent = FALSE`
- Optionally add an `IsDeleted` flag

**Advantages:**
- Preserves complete history
- Can still analyze historical data
- Supports compliance and audit requirements

### Option 2: Hard Delete

**Physically remove the record:**
- Execute `DELETE FROM table WHERE...`

**Disadvantages:**
- Loses all history
- Breaks referential integrity in fact tables
- Not recommended for dimensional tables

### Best Practice for SCD Type 2

**Never physically delete dimension records.** Instead, close the current record by setting `ValidTo` and `IsCurrent = FALSE`. This maintains referential integrity with your fact tables.

In [0]:
-- Carol closes her account on April 5, 2026
-- We perform a "soft delete" by closing her current record

UPDATE Customer_Type2
SET ValidTo = '2026-04-05',
    IsCurrent = FALSE
WHERE CustomerID = 3 AND IsCurrent = TRUE;

-- View all records including "deleted" ones
SELECT CustomerID, Name, SubscriptionTier, ValidFrom, ValidTo, IsCurrent
FROM Customer_Type2
ORDER BY CustomerID, ValidFrom;

-- Carol's record still exists but is marked as inactive

In [0]:
-- Get only ACTIVE customers
SELECT CustomerID, Name, SubscriptionTier
FROM Customer_Type2
WHERE IsCurrent = TRUE;

-- Get DELETED/INACTIVE customers (closed accounts)
SELECT CustomerID, Name, SubscriptionTier, ValidFrom, ValidTo
FROM Customer_Type2
WHERE IsCurrent = FALSE;

## SCD Type 3: Keep Previous Value (Limited History)

### What Is It?

**Type 3** means: **Store the current value AND the previous value in separate columns.**

Instead of creating new rows (Type 2) or overwriting (Type 1), you add columns like:
- `Current_SubscriptionTier`
- `Previous_SubscriptionTier`
- `Tier_EffectiveDate`

### When to Use Type 3

✅ **Use Type 3 when:**
- You only need to track **one level** of history (current + previous)
- Queries need to compare current vs. previous state easily
- Example: "Show all customers who downgraded from Premium"

❌ **Don't Use Type 3 when:**
- You need complete history (use Type 2 instead)
- Attributes change frequently (columns proliferate)

### Visual Example

```
Type 3 Table Structure:

CustomerID | Name  | Current_Tier | Previous_Tier | Tier_Changed_Date
1          | Alice | Premium      | Basic         | 2026-04-01
2          | Bob   | Premium      | NULL          | NULL
```

### Limitations

- **Only tracks ONE previous value** (what if Alice had 5 tier changes?)
- **Schema changes required** when adding new attributes to track
- **Not recommended** for most modern data warehouses
- Type 2 is generally preferred for flexibility

---

**Note:** We won't implement Type 3 in this notebook because Type 2 is the industry-standard approach for tracking dimensional history. Type 3 is mentioned for completeness and exam preparation.

## Production Pattern: Combined Type 1 + Type 2 in Single MERGE

### The Challenge

In real-world scenarios:
- Some attributes should use **Type 1** (overwrite): Email corrections, phone updates
- Other attributes should use **Type 2** (history): Subscription tier, pricing tier
- Both happen in the **same table** and **same pipeline run**

### The Solution: Conditional MERGE Logic

We'll use a single `MERGE` statement with multiple `WHEN MATCHED` clauses:

1. **Type 1 Check**: If ONLY Type 1 attributes changed → UPDATE in place
2. **Type 2 Check**: If Type 2 attributes changed → Close current record
3. **New Records**: INSERT new customers
4. **Post-MERGE**: INSERT new Type 2 records for customers who had Type 2 changes

### Critical: NULL Handling

⚠️ **DANGER**: Standard equality operators handle NULLs incorrectly!

```sql
-- WRONG: This fails silently when comparing NULLs
WHERE target.Email <> source.Email

-- CORRECT: Use COALESCE or IS DISTINCT FROM
WHERE COALESCE(target.Email, '') <> COALESCE(source.Email, '')
-- OR
WHERE target.Email IS DISTINCT FROM source.Email
```

Without proper NULL handling, your updates will **silently skip records**.

In [0]:
-- Create the main dimension table (Type 1 + Type 2 hybrid)
DROP TABLE IF EXISTS Dim_Customer;

CREATE TABLE Dim_Customer (
  CustomerKey BIGINT GENERATED ALWAYS AS IDENTITY,  -- Surrogate key
  CustomerID INT,                                    -- Natural key
  Name STRING,
  Email STRING,                -- Type 1 attribute (overwrite)
  Phone STRING,                -- Type 1 attribute (overwrite)
  SubscriptionTier STRING,     -- Type 2 attribute (preserve history)
  ValidFrom DATE,              -- Type 2: When this version became active
  ValidTo DATE,                -- Type 2: When this version was superseded
  IsCurrent BOOLEAN,           -- Type 2: Is this the current record?
  PRIMARY KEY (CustomerKey)
);

-- Insert initial data
INSERT INTO Dim_Customer (CustomerID, Name, Email, Phone, SubscriptionTier, ValidFrom, ValidTo, IsCurrent)
VALUES
  (1, 'Alice Johnson', 'alcie@example.com', '555-0101', 'Basic', '2025-01-01', NULL, TRUE),    -- Typo in email
  (2, 'Bob Smith', 'bob@example.com', '555-0102', 'Premium', '2025-01-15', NULL, TRUE);

SELECT * FROM Dim_Customer ORDER BY CustomerID, ValidFrom;

In [0]:
-- Staging table: Incoming batch of customer updates
DROP TABLE IF EXISTS Stg_Customer;

CREATE TABLE Stg_Customer (
  CustomerID INT,
  Name STRING,
  Email STRING,
  Phone STRING,
  SubscriptionTier STRING,
  LoadDate DATE
);

-- Simulate incoming data:
-- Alice: Email correction (Type 1) AND Tier upgrade (Type 2)
-- Bob: No changes (will be skipped)
-- Charlie: New customer (INSERT)
INSERT INTO Stg_Customer VALUES
  (1, 'Alice Johnson', 'alice@example.com', '555-0101', 'Premium', '2026-04-01'),  -- Type 1 + Type 2
  (2, 'Bob Smith', 'bob@example.com', '555-0102', 'Premium', '2026-04-01'),        -- No change
  (3, 'Charlie Brown', 'charlie@example.com', '555-0103', 'Basic', '2026-04-01'); -- New customer

SELECT * FROM Stg_Customer ORDER BY CustomerID;

### Critical Step: Deduplicate Staging Data

**Problem**: What if the staging table has multiple updates for the same customer?

The MERGE statement will fail or produce unpredictable results!

**Solution**: Use a CTE with `ROW_NUMBER()` to keep only the latest record per customer.

This pattern is used extensively in Databricks production pipelines.

In [0]:
-- Production MERGE: Handles Type 1, Type 2, and new records

MERGE INTO Dim_Customer AS target
USING (
  -- Deduplicate staging data (keep latest record per customer)
  SELECT CustomerID, Name, Email, Phone, SubscriptionTier, LoadDate
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY CustomerID ORDER BY LoadDate DESC) AS rn
    FROM Stg_Customer
  )
  WHERE rn = 1
) AS source
ON target.CustomerID = source.CustomerID AND target.IsCurrent = TRUE

-- Case 1: Type 1 change ONLY (Email or Phone changed, but NOT SubscriptionTier)
WHEN MATCHED AND (
  COALESCE(target.Email, '') <> COALESCE(source.Email, '') OR
  COALESCE(target.Phone, '') <> COALESCE(source.Phone, '')
) AND (
  COALESCE(target.SubscriptionTier, '') = COALESCE(source.SubscriptionTier, '')
)
THEN UPDATE SET
  target.Email = source.Email,
  target.Phone = source.Phone

-- Case 2: Type 2 change (SubscriptionTier changed)
-- Close the current record by setting ValidTo and IsCurrent = FALSE
WHEN MATCHED AND (
  COALESCE(target.SubscriptionTier, '') <> COALESCE(source.SubscriptionTier, '')
)
THEN UPDATE SET
  target.ValidTo = source.LoadDate,
  target.IsCurrent = FALSE,
  -- Also apply any Type 1 updates to the closing record
  target.Email = source.Email,
  target.Phone = source.Phone

-- Case 3: New customer (not in dimension table)
WHEN NOT MATCHED
THEN INSERT (CustomerID, Name, Email, Phone, SubscriptionTier, ValidFrom, ValidTo, IsCurrent)
VALUES (source.CustomerID, source.Name, source.Email, source.Phone, 
        source.SubscriptionTier, source.LoadDate, NULL, TRUE);

-- View results after MERGE
SELECT * FROM Dim_Customer ORDER BY CustomerID, ValidFrom;

### Step 3: Insert New Type 2 Records

The MERGE statement closed Alice's old 'Basic' record (set `IsCurrent = FALSE`, `ValidTo = 2026-04-01`).

Now we need to **INSERT** her new 'Premium' record with:
- `ValidFrom = 2026-04-01`
- `ValidTo = NULL`
- `IsCurrent = TRUE`

In [0]:
-- Insert new Type 2 records for customers who had SubscriptionTier changes
INSERT INTO Dim_Customer (CustomerID, Name, Email, Phone, SubscriptionTier, ValidFrom, ValidTo, IsCurrent)
SELECT 
  source.CustomerID,
  source.Name,
  source.Email,
  source.Phone,
  source.SubscriptionTier,
  source.LoadDate AS ValidFrom,
  NULL AS ValidTo,
  TRUE AS IsCurrent
FROM (
  SELECT CustomerID, Name, Email, Phone, SubscriptionTier, LoadDate
  FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY CustomerID ORDER BY LoadDate DESC) AS rn
    FROM Stg_Customer
  )
  WHERE rn = 1
) AS source
INNER JOIN Dim_Customer AS target
  ON source.CustomerID = target.CustomerID
WHERE target.IsCurrent = FALSE  -- Only insert for customers whose current record was just closed
  AND COALESCE(target.SubscriptionTier, '') <> COALESCE(source.SubscriptionTier, '')
  AND target.ValidTo = source.LoadDate;  -- Ensure we match the record we just closed

-- View final results
SELECT * FROM Dim_Customer ORDER BY CustomerID, ValidFrom;

## Verification: What Just Happened?

Let's verify the results:

### Alice (CustomerID = 1)
- **Old Record**: Email corrected from 'alcie@...' to 'alice@...', Tier was 'Basic'
  - `ValidFrom = 2025-01-01`, `ValidTo = 2026-04-01`, `IsCurrent = FALSE`
- **New Record**: Email correct, Tier upgraded to 'Premium'
  - `ValidFrom = 2026-04-01`, `ValidTo = NULL`, `IsCurrent = TRUE`

### Bob (CustomerID = 2)
- **No changes**: His record remains unchanged (one row, still current)

### Charlie (CustomerID = 3)
- **New customer**: Inserted as a brand new record
  - `ValidFrom = 2026-04-01`, `ValidTo = NULL`, `IsCurrent = TRUE`

In [0]:
-- View complete results with all history
SELECT 
  CustomerKey,
  CustomerID,
  Name,
  Email,
  SubscriptionTier,
  ValidFrom,
  ValidTo,
  IsCurrent,
  CASE 
    WHEN IsCurrent = TRUE THEN 'CURRENT'
    WHEN ValidTo IS NOT NULL THEN 'HISTORICAL'
  END AS RecordStatus
FROM Dim_Customer
ORDER BY CustomerID, ValidFrom;

In [0]:
-- Alice's complete history (should have 2 records)
SELECT 
  CustomerKey,
  CustomerID,
  Name,
  Email,
  SubscriptionTier,
  ValidFrom,
  ValidTo,
  IsCurrent
FROM Dim_Customer
WHERE CustomerID = 1
ORDER BY ValidFrom;

-- We see:
-- Row 1: Basic tier from 2025-01-01 to 2026-04-01 (CLOSED)
-- Row 2: Premium tier from 2026-04-01 to NULL (CURRENT)
-- Email was corrected in BOTH records (Type 1 logic applied)

In [0]:
-- Summary statistics
SELECT 
  COUNT(*) AS TotalRecords,
  SUM(CASE WHEN IsCurrent = TRUE THEN 1 ELSE 0 END) AS CurrentRecords,
  SUM(CASE WHEN IsCurrent = FALSE THEN 1 ELSE 0 END) AS HistoricalRecords,
  COUNT(DISTINCT CustomerID) AS UniqueCustomers
FROM Dim_Customer;

-- Expected:
-- TotalRecords: 4 (Alice has 2, Bob has 1, Charlie has 1)
-- CurrentRecords: 3 (one per customer)
-- HistoricalRecords: 1 (Alice's old Basic record)
-- UniqueCustomers: 3

## Summary: The Four Things to Remember

### 1. WHAT
- **Type 1**: Overwrite old data (corrections, no history)
- **Type 2**: Preserve history with `ValidFrom`, `ValidTo`, `IsCurrent`
- **Type 3**: Store current + previous value in separate columns (limited use)
- **Type 2 Deletes**: Soft delete by closing records, never hard delete

### 2. WHY
- A single `MERGE` statement prevents massive CPU/memory spikes of manual UPDATE pipelines
- At scale, separate UPDATE + INSERT statements cause:
  - Excessive database scans
  - DTU spikes in Azure SQL
  - Pipeline failures from duplicate keys
- Type 2 maintains referential integrity with fact tables

### 3. WHEN

**Use Type 1 when:**
- Correcting data quality issues (typos, formatting)
- Historical state is irrelevant
- NO compliance requirements for audit trails

**Use Type 2 when:**
- Analysts need point-in-time reporting
- Compliance mandates audit trails
- Revenue/metrics tied to dimension state at specific times

**DON'T use Type 2 for:**
- Rapidly changing attributes (LastLoginDate, PageViews)
- Creates "Rapidly Changing Dimension" anti-pattern
- Table bloat with millions of unnecessary rows

### 4. HOW

**Production Pattern:**
1. **Deduplicate staging data** using `ROW_NUMBER()` over `PARTITION BY natural_key`
2. **Single MERGE** with conditional logic:
   - Type 1 changes: `UPDATE` in place
   - Type 2 changes: Close current record (`ValidTo`, `IsCurrent = FALSE`)
   - New records: `INSERT`
3. **Post-MERGE INSERT** for new Type 2 records
4. **Critical**: Use `COALESCE()` or `IS DISTINCT FROM` for NULL-safe comparisons

---

## Common Pitfalls

### 1. NULL Comparison Failures
```sql
-- ❌ WRONG: Silently skips NULL comparisons
WHERE target.Email <> source.Email

-- ✅ CORRECT: NULL-safe comparison
WHERE COALESCE(target.Email, '') <> COALESCE(source.Email, '')
-- OR
WHERE target.Email IS DISTINCT FROM source.Email
```

### 2. Duplicate Keys in Staging
- **Problem**: Multiple updates for same customer in staging table
- **Solution**: Always deduplicate using `ROW_NUMBER() OVER (PARTITION BY key ORDER BY timestamp DESC)`

### 3. Hard Deletes in Type 2
- **Problem**: Physical DELETE breaks fact table foreign keys
- **Solution**: Soft delete by setting `ValidTo` and `IsCurrent = FALSE`

### 4. Not Applying Type 1 Updates to Closing Records
- **Problem**: Type 1 attributes (email) not updated when closing Type 2 record
- **Solution**: Apply Type 1 updates in the Type 2 closing UPDATE statement

---

## Performance Considerations

### Databricks Delta Lake Optimizations
1. **Z-Ordering**: `OPTIMIZE table_name ZORDER BY (natural_key, IsCurrent)`
2. **Data Skipping**: Delta automatically skips irrelevant files using min/max statistics
3. **MERGE Performance**: Delta Lake MERGE is optimized for SCD workloads

### Azure SQL Database Optimizations
1. **Indexes**: Create clustered index on `CustomerKey`, non-clustered on `(CustomerID, IsCurrent)`
2. **Partitioning**: Consider partitioning by date range for very large dimensions
3. **Statistics**: Keep statistics up-to-date with `UPDATE STATISTICS`

---

## Resources and References

### Databricks Documentation
- [Introducing SQL Scripting in Databricks (Part 2)](https://www.databricks.com/blog/introducing-sql-scripting-databricks-part-2)
- [Implementing Dimensional Data Warehouse in Databricks SQL (Part 3)](https://www.databricks.com/blog/implementing-dimensional-data-warehouse-databricks-sql-part-3)
- [How to Implement SCDs When You Have Duplicates](https://community.databricks.com/t5/technical-blog/how-to-implement-slowly-changing-dimensions-when-you-have/ba-p/40568)

### Exam Preparation
- **Azure Data Engineer Associate (DP-203)**: Domain 2 tests SCD Type 1 vs Type 2 vs Type 3
- **Interview Question**: "Why might an SCD MERGE fail?"
  - Answer: Duplicate updates for the same key in staging data

---

## The Databricks Advantage

This same pattern works identically in:
- **Databricks Delta Lake**: Using `MERGE INTO` with Delta tables
- **Azure SQL Database**: Using T-SQL `MERGE`
- **Snowflake**: Using `MERGE` statements

The SQL syntax is portable across modern data platforms!

## Next Steps

To implement this in your own data pipeline:

1. **Identify Your Attributes**
   - Which columns should be Type 1 (overwrite)?
   - Which columns should be Type 2 (history)?

2. **Add Type 2 Columns**
   - `ValidFrom DATE`
   - `ValidTo DATE`
   - `IsCurrent BOOLEAN`

3. **Build Your MERGE Logic**
   - Start with the template in this notebook
   - Customize the Type 1 vs Type 2 conditions
   - Add your specific business rules

4. **Test Thoroughly**
   - Test with duplicate staging records
   - Test with NULL values
   - Test Type 1 only, Type 2 only, and combined changes
   - Test new inserts and soft deletes

5. **Monitor Performance**
   - Track MERGE execution time
   - Monitor table growth (especially for Type 2)
   - Optimize with indexes, partitioning, or Z-ordering

6. **Automate**
   - Schedule your pipeline with Databricks Jobs
   - Add data quality checks
   - Implement alerting for failures

---

**Remember**: This pattern scales from thousands to billions of rows. It's the same logic used by Databricks architects at massive scale.

**Happy data engineering!** 🚀